# Example 2.1 / 1: Separating Model and Instance

Companion notebook for **2.1 Data Files and the `data` Statement** in the AMPL lab Documentation, section "Separating Model and Instance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Separating Model and Instance

The model below declares its required data. In `data.py` and `run.py`, `ampl` is an `AMPL`
object that has already read `model.mod`. Direct assignment is useful when Python owns the data;
`read_data()` is preferable when an existing `.dat` file owns it.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param profit {PROD};
param avail >= 0;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["avail"] = 40

In [5]:
# run.py
print(ampl.get_value("card(PROD)"))
print(ampl.param["avail"].value())

2
40


### Pure AMPL: data.dat and model.run

The same instance in `data.dat`, followed by AMPL commands:

In [6]:
%%writefile data.dat
set PROD := bands coils;
param profit := bands 25  coils 30;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display card(PROD), avail;

card(PROD) = 2
avail = 40

